# Step 04: Data Cleaning and Modeling Data Preparation
## MandiMitra ML Pipeline: Maharashtra Rice Mandi Time Series (2024–2026)

**Scope & Rules:**
- Crop: Rice | State: Maharashtra
- Strictly preserve original combined CSV (`data/processed/maharashtra_2024_2026_combined.csv`)
- No synthetic data, no arrival quantities
- No interpolation, forward-filling, or calendar resampling
- Clean types, strip whitespaces, validate price consistency
- Flag outliers with robust market-level statistics without deleting original values
- Enforce strict coverage threshold ($\ge 500$ daily observations) for reliable ML modeling



### Setup and Imports


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure root in sys.path
BASE_DIR = Path("..").resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.data_cleaning import (
    load_dataset,
    clean_column_names,
    clean_string_values,
    convert_price_columns,
    convert_date,
    check_price_consistency,
    investigate_and_remove_exact_duplicates,
    detect_outliers,
    evaluate_market_coverage,
    sort_and_verify_time_series,
    analyze_missing_dates,
)

RAW_COMBINED_PATH = BASE_DIR / "data" / "processed" / "maharashtra_2024_2026_combined.csv"
CLEAN_OUTPUT_PATH = BASE_DIR / "data" / "processed" / "maharashtra_rice_modeling_clean.csv"
COVERAGE_REPORT_PATH = BASE_DIR / "outputs" / "market_coverage_report.csv"



---
### 1. Load Data
Load combined Maharashtra Agmarknet dataset without modifying the source file.



In [2]:
df_raw = load_dataset(RAW_COMBINED_PATH)
print(f"Original Row Count: {len(df_raw):,}")
print(f"Original Column Count: {len(df_raw.columns)}")
display(df_raw.head(3))



Loaded dataset: maharashtra_2024_2026_combined.csv with 8,909 rows and 12 columns.
Original Row Count: 8,909
Original Column Count: 12


,State/UT,District,Market,Commodity Group,Commodity,Variety,Grade,Min Price,Max Price,Modal Price,Price Unit,Price Date
0,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,500.00","6,500.00","5,500.00",Rs./Quintal,2024-02-24
1,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,300.00","6,000.00","5,100.00",Rs./Quintal,2024-02-21
2,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,200.00","6,000.00","5,000.00",Rs./Quintal,2024-02-20


---
### 2. Clean Column Names
Strip leading and trailing whitespace characters from all column headers.



In [3]:
df = clean_column_names(df_raw)
print("Cleaned column names:")
print(list(df.columns))



Cleaned column names:
['State/UT', 'District', 'Market', 'Commodity Group', 'Commodity', 'Variety', 'Grade', 'Min Price', 'Max Price', 'Modal Price', 'Price Unit', 'Price Date']


---
### 3. Clean String Values
Strip leading and trailing whitespace from all categorical textual attributes.



In [4]:
df = clean_string_values(df)
print("String columns trimmed. Sample record:")
display(df.iloc[0:1])



String columns trimmed. Sample record:


,State/UT,District,Market,Commodity Group,Commodity,Variety,Grade,Min Price,Max Price,Modal Price,Price Unit,Price Date
0,Maharashtra,Ahilyanagar,APMC Karjat,Cereals,Rice,Other,FAQ,"4,500.00","6,500.00","5,500.00",Rs./Quintal,2024-02-24


---
### 4. Convert Price Columns to Numeric Float
Strip thousands separators (commas) and parse `Min Price`, `Max Price`, and `Modal Price` into floats.



In [5]:
df, non_convertible = convert_price_columns(df)
print("Non-convertible count per price column:")
for col, cnt in non_convertible.items():
    print(f"  - {col}: {cnt}")

print("\nPrice column summary statistics:")
display(df[['Min Price', 'Modal Price', 'Max Price']].describe().round(2))



Non-convertible count per price column:
  - Min Price: 0
  - Max Price: 0
  - Modal Price: 0

Price column summary statistics:


,Min Price,Modal Price,Max Price
count,8909.00,8909.00,8909.00
mean,3453.78,4223.31,5083.46
std,1165.56,1462.00,2030.72
min,100.00,280.00,280.00
25%,2500.00,3500.00,3800.00
50%,3500.00,4150.00,4800.00
75%,4100.00,5000.00,6000.00
max,8500.00,11100.00,15000.00


---
### 5. Convert Date
Convert `Price Date` to pandas datetime and verify bounded range [2024-01-01 to 2026-09-03].



In [6]:
df, is_valid_range = convert_date(df)
min_d = df['Price Date'].min().strftime('%Y-%m-%d')
max_d = df['Price Date'].max().strftime('%Y-%m-%d')
print(f"Date Range: {min_d} to {max_d}")
print(f"All dates strictly within [2024-01-01, 2026-09-03]: {is_valid_range}")



Date Range: 2024-01-01 to 2026-09-03
All dates strictly within [2024-01-01, 2026-09-03]: True


---
### 6. Price Consistency Verification
Verify logical business invariant: $\text{Min Price} \le \text{Modal Price} \le \text{Max Price}$.



In [7]:
n_viol, pct_viol, viol_df = check_price_consistency(df)
print(f"Price Invariant Violations: {n_viol} ({pct_viol:.2f}%)")
if n_viol > 0:
    display(viol_df.head())
else:
    print("[PASSED] Consistency condition strictly satisfied across all records.")



Price Invariant Violations: 0 (0.00%)
[PASSED] Consistency condition strictly satisfied across all records.


---
### 7. Duplicates Investigation and Exact Duplicate Removal
Investigate multi-recordings on `Market + Variety + Grade + Price Date`. Remove only exact duplicate rows.



In [8]:
df, key_dups_count, exact_removed = investigate_and_remove_exact_duplicates(df)
print(f"Observations sharing duplicate (Market, Variety, Grade, Price Date): {key_dups_count}")
print(f"Exact Duplicate Rows Removed: {exact_removed}")



Observations sharing duplicate (Market, Variety, Grade, Price Date): 0
Exact Duplicate Rows Removed: 0


---
### 8. Outlier Detection with Robust Statistics
Calculate market-level IQR bounds ($Q_1 - 1.5 \times \text{IQR}$ to $Q_3 + 1.5 \times \text{IQR}$) with $3\sigma$ fallback for zero-IQR sticky pricing.
Flag outliers in `outlier_flag` without modifying original prices.



In [9]:
df, total_outliers = detect_outliers(df)
print(f"Total Outliers Flagged across full dataset: {total_outliers:,} (out of {len(df):,} records)")

# Display sample flagged outliers
outlier_samples = df[df['outlier_flag'] == 1].sort_values('Modal Price')
print("\nSample Flagged Outliers (Lowest and Highest Modal Prices):")
display(pd.concat([outlier_samples.head(3), outlier_samples.tail(3)])[['Market', 'Variety', 'Grade', 'Price Date', 'Min Price', 'Modal Price', 'Max Price', 'outlier_flag']])



Total Outliers Flagged across full dataset: 485 (out of 8,909 records)

Sample Flagged Outliers (Lowest and Highest Modal Prices):


,Market,Variety,Grade,Price Date,Min Price,Modal Price,Max Price,outlier_flag
3952,APMC Palghar,1009 Kar,FAQ,2025-01-05,280.0,280.0,280.0,1
101,APMC Bhandara,Other,FAQ,2024-08-23,1300.0,1300.0,1300.0,1
2986,APMC Bhandara,Other,Local,2025-04-28,1800.0,1800.0,1800.0,1
7282,APMC Pune,Other,Local,2026-06-02,8400.0,9900.0,11400.0,1
3976,APMC Pune,Other,Local,2025-12-31,7300.0,9900.0,12500.0,1
7305,APMC Pune,Other,Local,2026-04-24,8500.0,10000.0,11500.0,1


---
### 9. Market / Variety / Grade Coverage Filtering
Filter time series to retain only groups with $\ge 500$ observations to guarantee statistically viable series.



In [10]:
coverage_report, modeling_df = evaluate_market_coverage(df, min_observations=500)
coverage_report.to_csv(COVERAGE_REPORT_PATH, index=False)

print(f"Total Groups Evaluated: {len(coverage_report)}")
print(f"Retained Groups (>= 500 obs): {coverage_report['Retained'].sum()}")
print(f"Excluded Groups (< 500 obs): {(~coverage_report['Retained']).sum()}")

print("\nRetained Groups:")
display(coverage_report[coverage_report['Retained']])

print("\nTop Excluded Groups (Near Threshold):")
display(coverage_report[~coverage_report['Retained']].head(5))



Total Groups Evaluated: 55
Retained Groups (>= 500 obs): 3
Excluded Groups (< 500 obs): 52

Retained Groups:


,Market,Variety,Grade,Observation Count,Retained
0,APMC Alibagh,Other,Local,575,True
1,APMC Murud,Other,Local,575,True
2,APMC Palghar,1009 Kar,Local,537,True



Top Excluded Groups (Near Threshold):


,Market,Variety,Grade,Observation Count,Retained
3,APMC Mangaon,Other,Local,496,False
4,APMC VASAI,1009 Kar,Local,433,False
5,APMC Solapur,Other,Local,432,False
6,APMC Ulhasnagar,Other,Local,391,False
7,APMC Alibagh,Other,FAQ,375,False


---
### 10. Chronological Sorting
Ensure each retained time series is sorted chronologically by `Price Date`.



In [11]:
modeling_df = sort_and_verify_time_series(modeling_df)
print(f"Modeling Dataset Shape: {modeling_df.shape[0]:,} rows x {modeling_df.shape[1]} columns")
display(modeling_df.head(5))



Modeling Dataset Shape: 1,687 rows x 13 columns


,State/UT,District,Market,Commodity Group,Commodity,Variety,Grade,Min Price,Max Price,Modal Price,Price Unit,Price Date,outlier_flag
0,Maharashtra,Raigad,APMC Alibagh,Cereals,Rice,Other,Local,3000.0,3500.0,3250.0,Rs./Quintal,2025-01-26,0
1,Maharashtra,Raigad,APMC Alibagh,Cereals,Rice,Other,Local,3000.0,3500.0,3250.0,Rs./Quintal,2025-01-27,0
2,Maharashtra,Raigad,APMC Alibagh,Cereals,Rice,Other,Local,3000.0,3500.0,3250.0,Rs./Quintal,2025-01-28,0
3,Maharashtra,Raigad,APMC Alibagh,Cereals,Rice,Other,Local,3000.0,3500.0,3250.0,Rs./Quintal,2025-01-29,0
4,Maharashtra,Raigad,APMC Alibagh,Cereals,Rice,Other,Local,3000.0,3500.0,3250.0,Rs./Quintal,2025-01-30,0


---
### 11. Missing Dates & Temporal Gap Analysis
Analyze trading cadence without interpolation or forward-filling.



In [12]:
gap_report = analyze_missing_dates(modeling_df)
display(gap_report)



,Market,Variety,Grade,Start Date,End Date,Observed Days,Calendar Span,Missing Days,Coverage Rate (%),Max Gap (Days)
0,APMC Alibagh,Other,Local,2025-01-26,2026-09-03,575,586,11,98.12,3
1,APMC Murud,Other,Local,2025-01-26,2026-09-03,575,586,11,98.12,3
2,APMC Palghar,1009 Kar,Local,2025-01-26,2026-09-03,537,586,49,91.64,6


---
### 12. Save Clean Modeling Dataset



In [13]:
modeling_df.to_csv(CLEAN_OUTPUT_PATH, index=False)
print(f"Saved cleaned modeling dataset to: {CLEAN_OUTPUT_PATH}")
print(f"Saved coverage report to: {COVERAGE_REPORT_PATH}")



Saved cleaned modeling dataset to: /Users/moksh/Desktop/MandiMitra-ML/data/processed/maharashtra_rice_modeling_clean.csv
Saved coverage report to: /Users/moksh/Desktop/MandiMitra-ML/outputs/market_coverage_report.csv


---
### 13. Final Cleaning & Readiness Summary



In [14]:
print("=" * 75)
print("FINAL DATA CLEANING & MODELING PREPARATION SUMMARY")
print("=" * 75)
print(f"Original Row Count                      : {len(df_raw):,}")
print(f"Final Modeling Row Count                : {len(modeling_df):,}")
print(f"Retained Market+Variety+Grade Groups    : {int(coverage_report['Retained'].sum())}")
print(f"Excluded Market+Variety+Grade Groups    : {int((~coverage_report['Retained']).sum())}")
print(f"Exact Duplicates Removed                : {exact_removed}")
print(f"Outlier Observations Flagged in Modeling: {int(modeling_df['outlier_flag'].sum())}")
print(f"Price Consistency Violations            : {n_viol}")
print(f"Missing Values in Modeling Dataset      : {modeling_df.isna().sum().sum()}")
print(f"Date Range of Modeling Dataset          : {modeling_df['Price Date'].min().strftime('%Y-%m-%d')} to {modeling_df['Price Date'].max().strftime('%Y-%m-%d')}")
print(f"Number of Retained Markets              : {modeling_df['Market'].nunique()}")
print(f"Number of Retained Varieties            : {modeling_df['Variety'].nunique()}")
print(f"Number of Retained Grades               : {modeling_df['Grade'].nunique()}")
print("=" * 75)



FINAL DATA CLEANING & MODELING PREPARATION SUMMARY
Original Row Count                      : 8,909
Final Modeling Row Count                : 1,687
Retained Market+Variety+Grade Groups    : 3
Excluded Market+Variety+Grade Groups    : 52
Exact Duplicates Removed                : 0
Outlier Observations Flagged in Modeling: 13
Price Consistency Violations            : 0
Missing Values in Modeling Dataset      : 0
Date Range of Modeling Dataset          : 2025-01-26 to 2026-09-03
Number of Retained Markets              : 3
Number of Retained Varieties            : 2
Number of Retained Grades               : 1
